# TTM × SIX G1 — Thử nghiệm khám phá với **Qwen thật**, không mở TEST

**Phạm vi:** 16 nhóm tình huống hư cấu mới × 4 câu, 48 TRAIN / 16 DEV. **Không đọc TinyStories, không dùng G1 TEST, không mount Drive, không chạy science G1 gốc.** Đây là *exploratory screening*, không phải G1 confirmatory. Chỉ chạy khi Colab Free cấp Tesla T4.

Nguồn model và checksum được khóa; hai Qwen load tuần tự. Sau chạy lấy `result.json` và gói lưu qua nút tải Colab ở cell cuối. Nếu môi trường không cấp T4 hoặc pin sai, STOP. Không tái chạy G1 technical one-shots.


In [ ]:
%pip install -q "transformers==5.22.1" "huggingface-hub==1.33.0" "tokenizers==0.23.3" "safetensors==0.8.0"

In [ ]:
# Offline source SHA check; this cell cannot read original G1 files.
import base64, hashlib, subprocess, sys, pathlib
SRC_BYTES = base64.b64decode('...')
EXPECTED_SHA = '9688deb1d1157c93fb7090e5b1c12ea4bc56ed0dedadb0597d84ceb4e68e82eb'
assert hashlib.sha256(SRC_BYTES).hexdigest() == EXPECTED_SHA, "SOURCE_SHA_MISMATCH"
SOURCE = pathlib.Path('/content/g1_explore_public_qwen_dev_v1.py')
if SOURCE.exists():
    assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == EXPECTED_SHA, "EXISTING_SOURCE_DIFFERS"
else:
    SOURCE.write_bytes(SRC_BYTES)
subprocess.run([sys.executable, str(SOURCE), '--mode', 'qa'], check=True)
print('EXPLORATORY_SOURCE_SHA256='+EXPECTED_SHA)


In [ ]:
# A genuine Qwen experiment. Execution requires interactive Colab Free Tesla T4.
import subprocess, sys
subprocess.run([sys.executable, '/content/g1_explore_public_qwen_dev_v1.py',
                '--mode', 'run', '--out', '/content/g1_explore_public_qwen_dev_v1'], check=True)


In [ ]:
# Download ONLY the new exploratory artifacts; do not mount Google Drive.
import pathlib, zipfile, hashlib
from google.colab import files
out = pathlib.Path('/content/g1_explore_public_qwen_dev_v1')
assert (out/'result.json').exists(), 'NO_REAL_MODEL_RESULT_YET'
zip_path = pathlib.Path('/content/TTM_SIX_G1_EXPLORE_REAL_QWEN_DEV_RESULTS.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/g1_explore_public_qwen_dev_v1.py', arcname='g1_explore_public_qwen_dev_v1.py')
    for p in sorted(out.iterdir()):
        if p.is_file(): z.write(p, arcname='results/'+p.name)
print('EXPLORATORY_RESULT_ZIP_SHA256='+hashlib.sha256(zip_path.read_bytes()).hexdigest())
print((out/'result.json').read_text('utf-8')[:16000])
files.download(str(zip_path))
